# Step 8c: where the deepfake evidence survives consumer processing, a frozen-detector back-end, and automatic processing-aware calibration

**Why:** Steps 8 and 8b changed the detector's weights (LoRA) and both times it forgot real-world fakes (In-the-Wild dev EER 4 -> 9.5%).
Here the detector is **never changed**. We run it once, keep the time-average of each of its 25 layers for every clip, and train small models on top.

**What this notebook measures and builds:**
1. **Layer analysis:** one linear probe per layer, so we can see *which layers still carry the deepfake evidence* after noise suppression, Bluetooth and Opus.
2. **Back-end + fusion:** a learned weighting of all 25 layers plus a linear layer, fused with the original head as `(1-w) * original + w * back-end`.
   `w = 0` is exactly the original model, and `w` is chosen on the dev sets, so the original can only be replaced if the dev data say so.
3. **Processing-aware calibration ("automatic line profile"):** a second small probe recognises which processing the audio went through (9 classes).
   Each processing gets its own threshold, calibrated on held-out speakers. This targets score *shifts* such as Opus, where 32.7% of real voices were flagged.

The test sets are the same unseen sets as in Step 8.

**Inputs:** GPU T4 x2, Internet ON, secret `HF_TOKEN`; attach your Step 4 notebook (Add Input -> Your Work) to reuse `step3_data.zip`.
**Outputs (small, please download):** `step8c_results.csv`, `step8c_layers.csv`, `step8c_fusion.csv`, `step8c_shift.csv`, `step8c_summary.json`, `step8c_backend.npz`.
**Large, download only when asked:** `voiceshield_layers.onnx` (the same detector, with the 25 layer averages as an extra output, for the live server).


In [ ]:
# CELL 1: installs (torchao is removed: Kaggle ships an old torchao that new peft versions refuse to load)
!pip uninstall -y -q onnxruntime torchao
!pip install -q onnxruntime-gpu noisereduce datasets peft onnx
!ffmpeg -version | head -n 1


In [ ]:
# CELL 2: setup + audio helpers
import os, re, glob, json, zipfile, tempfile, random, time, shutil
import numpy as np, pandas as pd, librosa, soundfile as sf, torch
import torch.nn as nn, torch.nn.functional as F
import pyarrow.parquet as pq
import noisereduce as nr
from scipy.signal import butter, sosfilt, resample_poly
from huggingface_hub import login, HfApi, hf_hub_download, snapshot_download
from datasets import load_dataset, Audio
from sklearn.metrics import roc_auc_score, roc_curve

try:                                   # Kaggle: Add-ons -> Secrets -> HF_TOKEN
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN, ON_COLAB = UserSecretsClient().get_secret("HF_TOKEN"), False
except ImportError:                    # Google Colab: key icon on the left -> HF_TOKEN, notebook access ON
    from google.colab import userdata, drive
    HF_TOKEN, ON_COLAB = userdata.get("HF_TOKEN"), True
login(token=HF_TOKEN)
api = HfApi()
random.seed(0); np.random.seed(0); torch.manual_seed(0)

SR, T   = 16000, 64000                 # 4.0 s window, same as the ONNX model
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
if ON_COLAB:                           # results go to Google Drive, so a disconnect cannot lose them
    drive.mount("/content/drive")
    WORK, TMP = "/content/drive/MyDrive/SIH26104/step4", "/content/tmp"
else:                                  # WORK = saved Output (model, CSVs, JSON); TMP = scratch audio, not saved
    WORK, TMP = "/kaggle/working", "/kaggle/temp"
os.makedirs(WORK, exist_ok=True)
try:
    os.makedirs(TMP, exist_ok=True)
except OSError:
    TMP = tempfile.gettempdir()
N_EXTRA = 300                          # extra Kathbath real AND IndicSynth fake clips per language

def decode(audio_bytes, name_hint=""):
    ext = os.path.splitext(name_hint or "")[1] or ".wav"
    with tempfile.NamedTemporaryFile(suffix=ext, delete=False) as f:
        f.write(audio_bytes)
    try:
        wav, _ = librosa.load(f.name, sr=SR, mono=True)
    finally:
        os.remove(f.name)
    return wav.astype(np.float32)

def load_wav(path):
    wav, sr = sf.read(path, dtype="float32")
    if wav.ndim > 1:
        wav = wav.mean(1)
    if sr != SR:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=SR)
    return wav

def trim(wav):
    t, _ = librosa.effects.trim(wav, top_db=30)
    return t if len(t) > SR // 2 else wav

_PHONE_SOS = butter(4, [300, 3400], btype="band", fs=SR, output="sos")

def phone(wav):
    """Simulated phone line: 300-3400 Hz band, 8 kHz, G.711 mu-law, back to 16 kHz."""
    x = sosfilt(_PHONE_SOS, wav)
    x = resample_poly(x, 1, 2)
    x = x / (np.abs(x).max() + 1e-9) * 0.9
    mu = 255.0
    y = np.round(np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu) * 127) / 127
    x = np.sign(y) * ((1 + mu) ** np.abs(y) - 1) / mu
    return resample_poly(x, 2, 1).astype(np.float32)

def colored_noise(n, beta):
    """beta 0 = white, 1 = pink, 2 = brown."""
    spec = np.fft.rfft(np.random.randn(n))
    f = np.fft.rfftfreq(n)
    f[0] = f[1]
    x = np.fft.irfft(spec / f ** (beta / 2), n)
    return x / (x.std() + 1e-9)

def add_noise(wav, snr_db):
    p = np.mean(wav ** 2) + 1e-12
    return wav + colored_noise(len(wav), random.choice([0, 1, 2])) * np.sqrt(p / 10 ** (snr_db / 10))

def lowpass(wav, cutoff):
    return sosfilt(butter(6, cutoff, btype="low", fs=SR, output="sos"), wav)

def crop(wav, random_start=False):
    if len(wav) < T:
        wav = np.tile(wav, T // max(len(wav), 1) + 1)
    s = random.randint(0, len(wav) - T) if random_start else 0
    return wav[s:s + T]

def augment(wav):
    """Identical channel effects for real and fake, so noise/bandwidth cannot reveal the label."""
    x = crop(wav, random_start=True).astype(np.float64)
    if random.random() < 0.25:                                  # denoise (makes some REAL clips clean too)
        x = nr.reduce_noise(y=x, sr=SR, stationary=True, prop_decrease=random.uniform(0.6, 1.0))
    if random.random() < 0.5:                                   # background noise (makes some FAKE clips noisy too)
        x = add_noise(x, random.uniform(5, 30))
    r = random.random()
    if r < 0.4:
        x = phone(x)                                            # phone line
    elif r < 0.6:
        x = lowpass(x, random.uniform(3500, 7000))              # reduced bandwidth
    return np.asarray(crop(x), dtype=np.float32)

def eer_auc(labels, p_fake):
    labels, p_fake = np.asarray(labels), np.asarray(p_fake)
    fpr, tpr, thr = roc_curve(labels, p_fake)
    fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr))
    return {"EER_%": 100 * (fpr[i] + fnr[i]) / 2, "AUC": roc_auc_score(labels, p_fake),
            "threshold_at_EER": float(thr[i]),
            "acc_at_0.5_%": 100 * ((p_fake >= 0.5).astype(int) == labels).mean()}

def acc_at(labels, p_fake, thr):
    return 100 * ((np.asarray(p_fake) >= thr).astype(int) == np.asarray(labels)).mean()

print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU - check settings")

# ======================= Step 8: consumer-DSP effects (what devices do to a REAL voice) =======================
import shutil as _shutil, subprocess as _sp
from scipy.signal import stft, istft
HAVE_FFMPEG = _shutil.which("ffmpeg") is not None
_BT_SOS = butter(4, [300, 3400], btype="band", fs=SR, output="sos")

def _fit(y, n):
    y = np.asarray(y, np.float64)
    return y[:n] if len(y) >= n else np.pad(y, (0, n - len(y)))

def agc(x, target_db=-23.0, max_db=20.0):
    """Automatic gain control: fast attack, slow release, soft limiter (OS / phone style)."""
    x = np.asarray(x, np.float64)
    n = 160; fr = len(x) // n
    if fr < 2:
        return x
    rms = np.sqrt(np.mean(x[:fr * n].reshape(fr, n) ** 2, axis=1) + 1e-10)
    want = np.clip(target_db - 20 * np.log10(rms), -max_db, max_db)
    g = np.empty(fr); cur = want[0]
    for i in range(fr):
        cur += (0.5 if want[i] < cur else 0.05) * (want[i] - cur)
        g[i] = cur
    y = x * np.interp(np.arange(len(x)), np.arange(fr) * n + n / 2, 10 ** (g / 20))
    return np.tanh(1.5 * y) / 1.5

def ns_gate(x):
    """Non-stationary spectral-gating noise suppression (leaves 'musical noise', like many suppressors)."""
    return _fit(nr.reduce_noise(y=np.asarray(x, np.float64), sr=SR, stationary=False,
                                prop_decrease=random.uniform(0.7, 1.0)), len(x))

def ns_specsub(x):
    """Spectral subtraction with over-subtraction, a gain floor and temporal smoothing (classic phone NS)."""
    x = np.asarray(x, np.float64)
    _, _, Z = stft(x, fs=SR, nperseg=512, noverlap=384)
    P = np.abs(Z) ** 2
    noise = np.percentile(P, 10, axis=1, keepdims=True)
    alpha, floor = random.uniform(1.5, 4.0), random.uniform(0.03, 0.15)
    G = np.sqrt(np.maximum(1 - alpha * noise / (P + 1e-12), floor ** 2))
    for t in range(1, G.shape[1]):
        G[:, t] = 0.6 * G[:, t] + 0.4 * G[:, t - 1]
    _, y = istft(Z * G, fs=SR, nperseg=512, noverlap=384)
    return _fit(y, len(x))

def bt_cvsd(x):
    """Bluetooth hands-free (HFP) narrowband voice: 8 kHz audio, CVSD delta modulation at 64 kbit/s."""
    x = np.asarray(x, np.float64)
    x8 = resample_poly(x, 1, 2)
    x8 = x8 / (np.abs(x8).max() + 1e-9) * 0.8
    u = resample_poly(x8, 8, 1)
    out = np.empty_like(u)
    est, step, hist = 0.0, 0.01, 0
    for i, v in enumerate(u):
        bit = 1 if v >= est else 0
        hist = ((hist << 1) | bit) & 15
        step = min(step + 0.01, 0.25) if hist in (0, 15) else max(step * 0.98, 0.002)
        est = 0.999 * est + (step if bit else -step)
        out[i] = est
    y = sosfilt(_BT_SOS, resample_poly(resample_poly(out, 1, 8), 2, 1))
    return _fit(y, len(x))

def bt_msbc(x):
    """Bluetooth wideband speech (mSBC-like): 16 kHz, band-limited to ~7 kHz, 8-bit companded."""
    y = lowpass(np.asarray(x, np.float64), 7000)
    y = y / (np.abs(y).max() + 1e-9) * 0.9
    q = np.round(np.sign(y) * np.log1p(255 * np.abs(y)) / np.log1p(255) * 127) / 127
    return np.sign(q) * ((1 + 255) ** np.abs(q) - 1) / 255

def opus(x, kbps=None):
    """Opus (WebRTC / WhatsApp calls) at a low bit rate, via ffmpeg; falls back to mSBC-like without ffmpeg."""
    if not HAVE_FFMPEG:
        return bt_msbc(x)
    kbps = kbps or random.choice([8, 12, 16, 24])
    pcm = (np.clip(x, -1, 1) * 32767).astype("<i2").tobytes()
    enc = _sp.run(["ffmpeg", "-loglevel", "quiet", "-f", "s16le", "-ar", "16000", "-ac", "1", "-i", "pipe:0",
                   "-c:a", "libopus", "-b:a", f"{kbps}k", "-f", "ogg", "pipe:1"], input=pcm, capture_output=True).stdout
    dec = _sp.run(["ffmpeg", "-loglevel", "quiet", "-i", "pipe:0", "-f", "s16le", "-ar", "16000", "-ac", "1",
                   "pipe:1"], input=enc, capture_output=True).stdout
    y = np.frombuffer(dec, "<i2").astype(np.float64) / 32768
    return _fit(y, len(x)) if len(y) > 100 else np.asarray(x, np.float64)

DSP = {"noise suppression": ns_gate, "spectral subtraction": ns_specsub, "auto-gain": agc,
       "auto-gain + noise suppression": lambda x: ns_specsub(agc(x)), "phone line (G.711)": phone,
       "Bluetooth CVSD": bt_cvsd, "Bluetooth mSBC": bt_msbc, "Opus": opus}
DSP_NAMES = list(DSP)

def augment(wav):
    """Step 8 augmentation, identical for real and fake: background noise, then ONE consumer-DSP effect."""
    x = crop(wav, random_start=True).astype(np.float64)
    if random.random() < 0.5:
        x = add_noise(x, random.uniform(5, 30))
    r = random.random()
    if r < 0.6:
        x = DSP[random.choice(DSP_NAMES)](x)
    elif r < 0.7:
        x = lowpass(x, random.uniform(3500, 7000))
    return np.asarray(crop(x), dtype=np.float32)

def rates_at(labels, p_fake, thr):
    """False-alarm rate on REAL clips and miss rate on FAKE clips at a fixed threshold (both in %)."""
    labels, p_fake = np.asarray(labels), np.asarray(p_fake)
    return (100 * float((p_fake[labels == 0] >= thr).mean()), 100 * float((p_fake[labels == 1] < thr).mean()))

print("ffmpeg (Opus) available:", HAVE_FFMPEG, "| DSP effects:", ", ".join(DSP_NAMES))


In [ ]:
# CELL 3: Step 3 data. Uses an attached Step 3 output if it is complete, otherwise rebuilds it here.
#         Rebuilding runs the Step 3 code with the same seed (same shards, speakers and splits): ~30-45 min.
INPUT  = "/kaggle/input"
S3_DIR = f"{TMP}/step3_data"
N_SHARDS, N_SPK, N_REF, N_PER = 6, 20, 40, 25          # same settings as Step 3

def remap(df, root):
    df = df.copy()
    df["path"] = [root + p[p.index("/step3_data") + len("/step3_data"):] for p in df.path]
    return df

def kathbath(lang, n_shards=N_SHARDS):
    files = sorted(f.path for f in api.list_repo_tree("ai4bharat/Kathbath", path_in_repo=lang,
                                                      repo_type="dataset", recursive=True)
                   if f.path.endswith(".parquet") and "train" in os.path.basename(f.path))
    pick = sorted({files[int(i)] for i in np.linspace(0, len(files) - 1, min(n_shards, len(files)))})
    print(f"{lang}: {len(files)} train shards, downloading {len(pick)}")
    return pd.concat([pq.read_table(hf_hub_download("ai4bharat/Kathbath", p, repo_type="dataset")).to_pandas()
                      for p in pick], ignore_index=True)

def rebuild_step3():
    """The Step 3 notebook's code: same-speaker kNN-VC clones of Kathbath speakers."""
    knn_vc = torch.hub.load("bshall/knn-vc", "knn_vc", prematched=True, trust_repo=True,
                            pretrained=True, device=DEVICE)
    random.seed(0)
    rows = []
    for lang in ["hindi", "telugu"]:
        df = kathbath(lang)
        counts = df.speaker_id.value_counts()
        ok = counts[counts >= N_REF + N_PER].index.tolist()
        gender = df.drop_duplicates("speaker_id").set_index("speaker_id")["gender"].astype(str).str.lower()
        fem = [s for s in ok if gender[s].startswith("f")]
        mal = [s for s in ok if not gender[s].startswith("f")]
        random.shuffle(fem); random.shuffle(mal)
        targets = fem[:N_SPK // 2] + mal[:N_SPK // 2]
        targets += [s for s in ok if s not in targets][:N_SPK - len(targets)]
        print(f"{lang}: {df.speaker_id.nunique()} speakers in shards, {len(ok)} with enough clips, using {len(targets)}")
        order = targets[:]; random.shuffle(order)
        n_tr, n_dev = int(0.7 * len(order)), int(0.15 * len(order))
        split = {s: ("train" if i < n_tr else "dev" if i < n_tr + n_dev else "test") for i, s in enumerate(order)}

        for k, spk in enumerate(targets):
            mine = df[df.speaker_id == spk].sample(frac=1, random_state=k)
            ref_rows, real_rows = mine.iloc[:N_REF], mine.iloc[N_REF:N_REF + N_PER]
            src_rows = df[df.speaker_id != spk].sample(N_PER, random_state=k)
            refs = [torch.from_numpy(trim(decode(a["bytes"], a.get("path")))) for a in ref_rows["audio_filepath"]]
            matching_set = knn_vc.get_matching_set(refs, vad_trigger_level=0)
            sdir = str(spk).replace("/", "_")
            for j, (_, r) in enumerate(real_rows.iterrows()):
                wav = trim(decode(r["audio_filepath"]["bytes"], r["audio_filepath"].get("path")))
                p = f"{S3_DIR}/{lang}/real/{sdir}/{j:03d}.flac"
                os.makedirs(os.path.dirname(p), exist_ok=True); sf.write(p, wav, SR)
                rows.append(dict(path=p, label=0, lang=lang, speaker=spk, gender=gender[spk],
                                 generator="bonafide", source_speaker=spk, split=split[spk], text=r["text"]))
            for j, (_, r) in enumerate(src_rows.iterrows()):
                src = torch.from_numpy(trim(decode(r["audio_filepath"]["bytes"], r["audio_filepath"].get("path"))))
                with torch.inference_mode():
                    out = knn_vc.match(knn_vc.get_features(src), matching_set, topk=4)
                wav = trim(out.detach().cpu().numpy().astype(np.float32))
                p = f"{S3_DIR}/{lang}/fake/{sdir}/{j:03d}.flac"
                os.makedirs(os.path.dirname(p), exist_ok=True); sf.write(p, wav, SR)
                rows.append(dict(path=p, label=1, lang=lang, speaker=spk, gender=gender[spk],
                                 generator="knnvc", source_speaker=r["speaker_id"], split=split[spk], text=r["text"]))
            print(f"  [{lang}] speaker {k + 1}/{len(targets)} ({split[spk]}) done")
        del df
    del knn_vc
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    meta = pd.DataFrame(rows)
    meta.to_csv(f"{S3_DIR}/metadata.csv", index=False)
    return meta

s3 = None
# 1) Step 3 notebook output attached as Kaggle input (only if every audio file made it into the output)
for meta_csv in sorted(glob.glob(f"{INPUT}/**/step3_data/metadata.csv", recursive=True)):
    cand = remap(pd.read_csv(meta_csv), os.path.dirname(meta_csv))
    n_ok = sum(os.path.exists(p) for p in cand.path)
    print(f"Attached Step 3 output: {meta_csv} -> {n_ok}/{len(cand)} audio files present")
    if n_ok == len(cand):
        s3 = cand
        break
# 2) step3_data.zip saved by an earlier run of THIS notebook (Google Drive on Colab, or attached on Kaggle)
if s3 is None:
    for zp in sorted(glob.glob(f"{INPUT}/**/step3_data.zip", recursive=True)) + glob.glob(f"{WORK}/step3_data.zip"):
        try:
            dest = f"{TMP}/step3_from_zip"
            shutil.rmtree(dest, ignore_errors=True)
            with zipfile.ZipFile(zp) as z:
                z.extractall(dest)
            cand = remap(pd.read_csv(f"{dest}/metadata.csv"), dest)
            if all(os.path.exists(p) for p in cand.path):
                print("Reusing saved Step 3 clips:", zp)
                s3 = cand
                break
        except Exception as e:
            print("Could not use", zp, "->", e)
# 3) otherwise rebuild, then save ONE zip file so a rerun can skip this
if s3 is None:
    print("Step 3 audio not attached (or incomplete), so rebuilding it here. About 30-45 min...")
    s3 = rebuild_step3()
    shutil.make_archive(f"{TMP}/step3_data", "zip", S3_DIR)
    shutil.copy(f"{TMP}/step3_data.zip", f"{WORK}/step3_data.zip")
    print("Saved", f"{WORK}/step3_data.zip", "(a rerun will reuse it)")

s3["source"] = "step3_kathbath_knnvc"
HELD_OUT = set(s3.loc[s3.split != "train", "speaker"].astype(str))   # dev/test speakers: never train on them
print(s3.groupby(["lang", "split", "label"]).size().unstack())


In [ ]:
# CELL 4: rebuild AntiDeepfake in PyTorch + check it matches the ONNX model
from transformers import Wav2Vec2Config, Wav2Vec2Model
from safetensors.torch import load_file

class Detector(nn.Module):
    """wav2vec 2.0 Large -> mean over time -> Linear(1024, 2). Output index 0 = FAKE, 1 = REAL."""
    def __init__(self, stable=True):
        super().__init__()
        cfg = Wav2Vec2Config(
            hidden_size=1024, num_hidden_layers=24, num_attention_heads=16, intermediate_size=4096,
            feat_extract_norm="layer", conv_bias=True, do_stable_layer_norm=stable,
            feat_extract_activation="gelu", hidden_act="gelu",
            num_conv_pos_embeddings=128, num_conv_pos_embedding_groups=16,
            hidden_dropout=0.1, attention_dropout=0.1, activation_dropout=0.0,
            feat_proj_dropout=0.1, layerdrop=0.0, apply_spec_augment=False,
            mask_time_prob=0.0, mask_feature_prob=0.0)
        self.ssl = Wav2Vec2Model(cfg)
        self.head = nn.Linear(1024, 2)

    def forward(self, wav):                                  # wav: (B, samples), 16 kHz
        wav = F.layer_norm(wav, wav.shape[-1:])              # per-utterance normalisation
        return self.head(self.ssl(wav).last_hidden_state.mean(1))

    @torch.no_grad()
    def p_fake(self, wav):
        return self.forward(wav).float().softmax(-1)[:, 0]

def fairseq_to_hf(sd, hf_keys):
    """Rename NII/fairseq wav2vec2 weights to Hugging Face Wav2Vec2Model names."""
    out = {}
    for k, v in sd.items():
        if not k.startswith("m_ssl.model."):
            continue
        k = k[len("m_ssl.model."):]
        if k.startswith(("quantizer.", "project_q.", "final_proj.", "mask_emb")):
            continue                                         # pre-training-only parts
        if k.startswith("post_extract_proj."):
            k = "feature_projection.projection." + k[len("post_extract_proj."):]
        elif k.startswith("layer_norm."):
            k = "feature_projection." + k
        k = re.sub(r"conv_layers\.(\d+)\.0\.", r"conv_layers.\1.conv.", k)
        k = re.sub(r"conv_layers\.(\d+)\.2\.1\.", r"conv_layers.\1.layer_norm.", k)
        k = k.replace("encoder.pos_conv.0.", "encoder.pos_conv_embed.conv.")
        k = k.replace(".self_attn_layer_norm.", ".layer_norm.").replace(".self_attn.", ".attention.")
        k = k.replace(".fc1.", ".feed_forward.intermediate_dense.").replace(".fc2.", ".feed_forward.output_dense.")
        if k not in hf_keys:                                 # newer PyTorch weight-norm naming
            k = (k.replace("conv.weight_g", "conv.parametrizations.weight.original0")
                  .replace("conv.weight_v", "conv.parametrizations.weight.original1"))
        out[k] = v
    return out

def load_antideepfake(sd, stable=True):
    m = Detector(stable)
    m.ssl.load_state_dict(fairseq_to_hf(sd, set(m.ssl.state_dict())), strict=True)
    m.head.weight.data.copy_(sd["proj_fc.weight"])
    m.head.bias.data.copy_(sd["proj_fc.bias"])
    return m.to(DEVICE).eval()

NII_SD = load_file(hf_hub_download("nii-yamagishilab/wav2vec-large-anti-deepfake", "model.safetensors"))

# Parity check against the ONNX model used in Steps 2-3
import onnxruntime as ort
if hasattr(ort, "preload_dlls"):
    ort.preload_dlls()
sess = ort.InferenceSession(hf_hub_download("SpeechAntiSpoofingBenchmarks/Wav2Vec2-Large-AntiDeepfake",
                                            "wav2vec2-large-antideepfake.onnx"),
                            providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
probe = s3[s3.split == "dev"].groupby("label").head(8)
x = np.stack([crop(load_wav(p)) for p in probe.path]).astype(np.float32)
xn = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-7)
onnx_logits = sess.run(None, {sess.get_inputs()[0].name: xn})[0]

for stable in (True, False):
    model = load_antideepfake(NII_SD, stable)
    with torch.no_grad():
        pt_logits = model(torch.from_numpy(x).to(DEVICE)).float().cpu().numpy()
    diff = float(np.abs(pt_logits - onnx_logits).max())
    soft = lambda l: np.exp(l[:, 0]) / np.exp(l).sum(1)
    print(f"stable_layer_norm={stable}: max logit diff vs ONNX = {diff:.4f}, "
          f"max p_fake diff = {np.abs(soft(pt_logits) - soft(onnx_logits)).max():.4f}")
    if diff < 0.5:                       # a wrong weight mapping gives differences of several units
        break
    del model; torch.cuda.empty_cache()
else:
    raise SystemExit("PyTorch model does not match ONNX - send these numbers to your mentor/Claude.")
del sess
print("PyTorch model matches ONNX. Parameters:", sum(p.numel() for p in model.parameters()) // 1_000_000, "M")


In [ ]:
# CELL 5: build train / dev / test data (more real speech than Step 4) + DSP test sets
N_REAL_EXTRA, N_FAKE_EXTRA = 450, 450        # per language
os.makedirs(f"{TMP}/step8_data", exist_ok=True)
extra = []

# 5a. Extra REAL Kathbath speech (train shards Step 3 did not use; dev/test speakers excluded)
for lang in ["hindi", "telugu"]:
    files = sorted(f.path for f in api.list_repo_tree("ai4bharat/Kathbath", path_in_repo=lang,
                                                      repo_type="dataset", recursive=True)
                   if f.path.endswith(".parquet") and "train" in os.path.basename(f.path))
    used = {files[int(i)] for i in np.linspace(0, len(files) - 1, min(6, len(files)))}
    pick = [f for f in files if f not in used][::max(1, (len(files) - len(used)) // 4)][:4]
    df = pd.concat([pq.read_table(hf_hub_download("ai4bharat/Kathbath", p, repo_type="dataset")).to_pandas()
                    for p in pick], ignore_index=True)
    df = df[~df.speaker_id.astype(str).isin(HELD_OUT)]
    df = df.sample(frac=1, random_state=1).groupby("speaker_id").head(12).head(N_REAL_EXTRA)
    for j, r in enumerate(df.itertuples()):
        p = f"{TMP}/step8_data/{lang}/kathbath_real/{j:04d}.flac"
        os.makedirs(os.path.dirname(p), exist_ok=True)
        sf.write(p, trim(decode(r.audio_filepath["bytes"], r.audio_filepath.get("path"))), SR)
        extra.append(dict(path=p, label=0, lang=lang, speaker=str(r.speaker_id), split="train",
                          source="kathbath_extra", generator="bonafide"))
    print(f"{lang}: {len(df)} extra real clips from {df.speaker_id.nunique()} speakers")
    del df

# 5b. IndicSynth FAKE speech (XTTS-v2 / VITS / FreeVC clones of Kathbath speakers)
for lang, cfg in [("hindi", "Hindi"), ("telugu", "Telugu")]:
    ds = load_dataset("vdivyasharma/IndicSynth", name=cfg, split="train", streaming=True)
    acol = next((k for k, v in (ds.features or {}).items() if isinstance(v, Audio)), "audio")
    ds = ds.cast_column(acol, Audio(decode=False)).shuffle(seed=1, buffer_size=2000)
    n, gens = 0, {}
    for row in ds:
        tgt = str(row.get("Target Speaker ID"))
        if tgt in HELD_OUT:
            continue
        p = f"{TMP}/step8_data/{lang}/indicsynth_fake/{n:04d}.flac"
        os.makedirs(os.path.dirname(p), exist_ok=True)
        sf.write(p, trim(decode(row[acol]["bytes"], row[acol].get("path"))), SR)
        g = row.get("Generative Model", "indicsynth")
        gens[g] = gens.get(g, 0) + 1
        extra.append(dict(path=p, label=1, lang=lang, speaker=f"isyn_{tgt}", split="train",
                          source="indicsynth", generator=g))
        n += 1
        if n >= N_FAKE_EXTRA:
            break
    print(f"{lang}: {n} IndicSynth fakes, generators: {gens}")

# 5c. Commercial TTS with the Step 4 split (Hume = dev, ElevenLabs + Kokoro = test, never trained on)
GS = snapshot_download("garystafford/deepfake-audio-detection", repo_type="dataset",
                       allow_patterns=["fake/*", "real/*"])
comm = []
for p in sorted(glob.glob(f"{GS}/fake/*.flac")) + sorted(glob.glob(f"{GS}/real/*.flac")):
    name = os.path.basename(p)
    if "/fake/" in p.replace("\\", "/"):
        plat, label = name.split("_")[0], 1
        split = "train" if plat in ("po", "sp", "lv") else "dev" if plat == "hu" else "test"
    else:
        plat, label = name[:7], 0
        split = ("test" if plat in ("yt_0009", "yt_0010", "yt_0002") else
                 "dev" if plat in ("yt_0013", "yt_0001", "yt_0003", "yt_0007") else "train")
    comm.append(dict(path=p, label=label, lang="english", speaker=plat, split=split,
                     source="commercial_tts", generator=plat if label else "bonafide"))
comm = pd.DataFrame(comm)

train_df = pd.concat([s3[s3.split == "train"], pd.DataFrame(extra), comm[comm.split == "train"]],
                     ignore_index=True)
train_df.to_csv(f"{WORK}/step8_train.csv", index=False)
print("\nTRAINING SET:"); print(train_df.groupby(["source", "label"]).size().unstack(fill_value=0))

# 5d. In-the-Wild: dev sample (checkpoint selection) and a separate test sample
zp = hf_hub_download("mueller91/In-The-Wild", "release_in_the_wild.zip", repo_type="dataset")
zf = zipfile.ZipFile(zp)
meta_name = next(n for n in zf.namelist() if n.endswith("meta.csv"))
root = os.path.dirname(meta_name)
itw = pd.read_csv(zf.open(meta_name))
itw["y"] = itw["label"].astype(str).str.lower().str.contains("spoof").astype(int)
itw_dev = itw.groupby("y").sample(n=200, random_state=1)
itw_test = itw.drop(itw_dev.index).groupby("y").sample(n=300, random_state=2)
def itw_set(d):
    return ([trim(decode(zf.read(f"{root}/{f}" if root else f), f)) for f in d.file], d.y.tolist())

def disk_set(d):
    return [trim(load_wav(p)) for p in d.path], d.label.tolist()

def dsp_set(base, effect, seed):
    """Same clips with background noise (15 dB) and ONE effect, deterministic (seeded)."""
    random.seed(seed); np.random.seed(seed)
    w, y = base
    return [np.asarray(DSP[effect](add_noise(np.asarray(x, np.float64), 15.0)), np.float32) for x in w], y

dev3, test3 = disk_set(s3[s3.split == "dev"]), disk_set(s3[s3.split == "test"])
comm_test = disk_set(comm[comm.split == "test"])
itw_test_set = itw_set(itw_test)
DEV = {
    "held-out speakers, clean": dev3,
    "held-out speakers, phone": ([phone(w) for w in dev3[0]], dev3[1]),
    "Commercial, unseen Hume": disk_set(comm[comm.split == "dev"]),
    "In-the-Wild (dev sample)": itw_set(itw_dev),
    "held-out speakers, noise suppression": dsp_set(dev3, "noise suppression", 11),
    "held-out speakers, auto-gain + noise suppression": dsp_set(dev3, "auto-gain + noise suppression", 12),
    "held-out speakers, Bluetooth CVSD": dsp_set(dev3, "Bluetooth CVSD", 13),
    "held-out speakers, Opus": dsp_set(dev3, "Opus", 14),
}
TEST = {
    "test speakers, clean": test3,
    "test speakers, phone": ([phone(w) for w in test3[0]], test3[1]),
    "test speakers, noise suppression": dsp_set(test3, "noise suppression", 21),
    "test speakers, spectral subtraction": dsp_set(test3, "spectral subtraction", 22),
    "test speakers, auto-gain + noise suppression": dsp_set(test3, "auto-gain + noise suppression", 23),
    "test speakers, Bluetooth CVSD": dsp_set(test3, "Bluetooth CVSD", 24),
    "test speakers, Bluetooth mSBC": dsp_set(test3, "Bluetooth mSBC", 25),
    "test speakers, Opus": dsp_set(test3, "Opus", 26),
    "ElevenLabs+Kokoro, clean": comm_test,
    "ElevenLabs+Kokoro, noise suppression": dsp_set(comm_test, "noise suppression", 27),
    "ElevenLabs+Kokoro, Opus": dsp_set(comm_test, "Opus", 28),
    "In-the-Wild (test sample)": itw_test_set,
    "In-the-Wild, noise suppression": dsp_set(itw_test_set, "noise suppression", 29),
}
for name, (w, y) in {**DEV, **TEST}.items():
    print(f"{name}: {y.count(0)} real, {y.count(1)} fake")


In [ ]:
# CELL 6: run the FROZEN detector once; keep, for every clip, the time-average of all hidden layers
# (CNN output + 24 transformer blocks) and the original model's score. Every later cell trains small models
# on these features, so the original detector itself never changes (no forgetting by construction).
import copy
random.seed(0); np.random.seed(0); torch.manual_seed(0)
CONDS = ["clean", "phone line (G.711)", "noise suppression", "spectral subtraction", "auto-gain",
         "auto-gain + noise suppression", "Bluetooth CVSD", "Bluetooth mSBC", "Opus"]
VIEWS, P_PROCESSED, FEAT_BS = 3, 0.85, 16          # 1 clean + 2 processed views of every training clip
model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False
NL, D = model.ssl.config.num_hidden_layers + 1, model.ssl.config.hidden_size

@torch.no_grad()
def layer_feats(x):
    """x: (B, T) -> layer averages (B, NL, D) as fp16 numpy, and the original score llr = logit_fake - logit_real."""
    with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
        out = model.ssl(F.layer_norm(x, x.shape[-1:]), output_hidden_states=True)
        # the last entry is taken from last_hidden_state (after the final layer norm), because transformers
        # versions differ on whether hidden_states[-1] is before or after that norm
        H = torch.stack([h.mean(1) for h in out.hidden_states[:-1]] + [out.last_hidden_state.mean(1)], 1)
    lo = model.head(H[:, -1].float())
    return H.half().cpu().numpy(), (lo[:, 0] - lo[:, 1]).float().cpu().numpy()

def set_feats(wavs):
    Hs, Ls = [], []
    for i in range(0, len(wavs), FEAT_BS):
        x = torch.from_numpy(np.stack([crop(w) for w in wavs[i:i + FEAT_BS]]).astype(np.float32)).to(DEVICE)
        h, l = layer_feats(x)
        Hs.append(h); Ls.append(l)
    return np.concatenate(Hs), np.concatenate(Ls)

# check: the last layer + the original head reproduce the detector's own output
xb = torch.from_numpy(np.stack([crop(w) for w in DEV["held-out speakers, clean"][0][:8]]).astype(np.float32)).to(DEVICE)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
    ref = model(xb).float()
diff = float(np.abs((ref[:, 0] - ref[:, 1]).cpu().numpy() - layer_feats(xb)[1]).max())
print(f"layer features + original head vs detector output: max llr difference {diff:.4f}")
assert diff < 0.1, "the feature path does not reproduce the detector"

# extra held-out sets so that every processing has its own calibration set
EXTRA = {f"held-out speakers, {c}": dsp_set(dev3, c, 31 + i)
         for i, c in enumerate(["spectral subtraction", "auto-gain", "Bluetooth mSBC"])}
SETS = {**DEV, **EXTRA, **TEST}
FE, t0 = {}, time.time()
for name, (w, y) in SETS.items():
    H, L = set_feats(w)
    FE[name] = (H, L, np.asarray(y))
print(f"features of {len(FE)} dev/calibration/test sets in {(time.time() - t0) / 60:.1f} min")

ORDER = ["auto-gain + noise suppression", "spectral subtraction", "noise suppression", "auto-gain",
         "Bluetooth CVSD", "Bluetooth mSBC", "Opus"]
def cond_of(name):
    """The processing a dev/test set was made with (In-the-Wild and commercial clips count as 'clean')."""
    for c in ORDER:
        if c in name:
            return c
    return "phone line (G.711)" if "phone" in name else "clean"

class ViewSet(torch.utils.data.Dataset):
    """View 0: clean crop. Views 1..: background noise (half the time) + one processing, kept as a label."""
    def __init__(self, df):
        self.items = [(p, int(l), v) for p, l in zip(df.path, df.label) for v in range(VIEWS)]
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label, v = self.items[i]
        x = crop(trim(load_wav(path)), random_start=True).astype(np.float64)
        c = 0
        if v > 0:
            if random.random() < 0.5:
                x = add_noise(x, random.uniform(5, 30))
            if random.random() < P_PROCESSED:
                c = random.randrange(1, len(CONDS))
                x = DSP[CONDS[c]](x)
        return torch.from_numpy(np.asarray(crop(x), np.float32)), label, c

def _seed_worker(_):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); random.seed(s)

views = ViewSet(train_df)
view_loader = torch.utils.data.DataLoader(views, batch_size=FEAT_BS, shuffle=False,
                                          num_workers=min(4, os.cpu_count() or 2), worker_init_fn=_seed_worker)
N = len(views)
Htr = np.empty((N, NL, D), np.float16); Ltr = np.empty(N, np.float32)
Ytr = np.empty(N, np.int64); Ctr = np.empty(N, np.int64)
k, t0 = 0, time.time()
for b, (x, y, c) in enumerate(view_loader):
    h, l = layer_feats(x.to(DEVICE))
    n = len(y)
    Htr[k:k + n], Ltr[k:k + n], Ytr[k:k + n], Ctr[k:k + n] = h, l, y.numpy(), c.numpy()
    k += n
    if b % 200 == 0:
        print(f"training views {k}/{N} ({(time.time() - t0) / 60:.1f} min)")
print(f"{N} training views ({len(train_df)} clips x {VIEWS}) in {(time.time() - t0) / 60:.1f} min; processing used:",
      dict(zip(CONDS, np.bincount(Ctr, minlength=len(CONDS)).tolist())))


In [ ]:
# CELL 7: where does the deepfake evidence survive consumer processing? One linear probe per layer,
# trained on the training views, EER on every dev and test set.
torch.manual_seed(0)
Xtr = torch.from_numpy(Htr).to(DEVICE)                       # fp16 on the GPU
ytr = torch.from_numpy(Ytr).float().to(DEVICE)               # 1 = fake
MU = torch.stack([Xtr[:, l].float().mean(0) for l in range(NL)])
SD = torch.stack([Xtr[:, l].float().std(0) for l in range(NL)])
SD = torch.maximum(SD, 0.01 * SD.median(1, keepdim=True).values) + 1e-6   # floor: near-constant dims stay harmless

def eer(y, s):
    return float(eer_auc(y, s)["EER_%"])

def fit_linear(x, y, steps=300):
    lin = nn.Linear(x.shape[1], 1).to(DEVICE)
    opt = torch.optim.AdamW(lin.parameters(), lr=1e-2, weight_decay=1e-2)
    for _ in range(steps):
        opt.zero_grad()
        F.binary_cross_entropy_with_logits(lin(x).squeeze(1), y).backward()
        opt.step()
    return lin.eval()

def torch_layer(H, l):
    return (torch.from_numpy(np.ascontiguousarray(H[:, l])).to(DEVICE).float() - MU[l]) / SD[l]

LAYER_NAMES = [f"layer {l:02d}" for l in range(NL)] + ["original head"]
rows = []
for l in range(NL):
    lin = fit_linear((Xtr[:, l].float() - MU[l]) / SD[l], ytr)
    with torch.no_grad():
        for name, (H, L, y) in FE.items():
            rows.append({"layer": LAYER_NAMES[l], "set": name,
                         "EER_%": eer(y, lin(torch_layer(H, l)).squeeze(1).cpu().numpy())})
for name, (H, L, y) in FE.items():
    rows.append({"layer": "original head", "set": name, "EER_%": eer(y, L)})
layers = pd.DataFrame(rows)
layers.to_csv(f"{WORK}/step8c_layers.csv", index=False)
piv = layers.pivot(index="layer", columns="set", values="EER_%").reindex(LAYER_NAMES)
piv["dev mean"] = piv[list(DEV)].mean(1)
piv["test mean"] = piv[list(TEST)].mean(1)
show = [k for k in ["held-out speakers, clean", "In-the-Wild (dev sample)", "held-out speakers, noise suppression",
                    "held-out speakers, Bluetooth CVSD", "held-out speakers, Opus"] if k in piv]
print("Linear-probe EER % per layer (layer 00 = CNN output, layer 24 = last block):")
print(piv[show + ["dev mean", "test mean"]].round(2).to_string())


In [ ]:
# CELL 8: our back-end: a learned weighting of ALL layers + a linear layer, on top of the frozen detector,
# fused with the original head. Fusion weight w = 0 is exactly the original model; w is chosen on dev.
class LayerProbe(nn.Module):
    def __init__(self, n_out):
        super().__init__()
        self.a = nn.Parameter(torch.zeros(NL))                  # layer weights (softmax)
        self.drop = nn.Dropout(0.2)
        self.lin = nn.Linear(D, n_out)
    def forward(self, H):
        z = (H.float() - MU) / SD
        return self.lin(self.drop((torch.softmax(self.a, 0)[None, :, None] * z).sum(1)))

@torch.no_grad()
def probe_out(probe, H, bs=512):
    probe.eval()
    return torch.cat([probe(torch.from_numpy(H[i:i + bs]).to(DEVICE)) for i in range(0, len(H), bs)]).float().cpu().numpy()

def train_probe(n_out, target, metric, epochs, lr=1e-3, bs=256):
    """Trains on the training views; keeps the epoch with the LOWEST metric(probe) on dev data."""
    torch.manual_seed(0)
    probe = LayerProbe(n_out).to(DEVICE)
    opt = torch.optim.AdamW(probe.parameters(), lr=lr, weight_decay=1e-2)
    best, best_ep, best_state, hist = float("inf"), 0, None, []
    for ep in range(1, epochs + 1):
        probe.train()
        for idx in torch.randperm(len(Xtr), device=DEVICE).split(bs):
            out = probe(Xtr[idx])
            loss = (F.binary_cross_entropy_with_logits(out.squeeze(1), target[idx]) if n_out == 1
                    else F.cross_entropy(out, target[idx]))
            opt.zero_grad(); loss.backward(); opt.step()
        m = metric(probe)
        hist.append(m)
        if m < best:
            best, best_ep, best_state = m, ep, copy.deepcopy(probe.state_dict())
    probe.load_state_dict(best_state)
    return probe.eval(), best_ep, hist

DEV_NAMES = list(DEV)
def dev_mean_eer(scores):
    return float(np.mean([eer(FE[k][2], scores[k]) for k in DEV_NAMES]))

det, det_ep, det_hist = train_probe(1, ytr, lambda p: dev_mean_eer({k: probe_out(p, FE[k][0])[:, 0] for k in DEV_NAMES}),
                                    epochs=30)
O = {k: L for k, (H, L, y) in FE.items()}                              # original score (positive = fake)
P = {k: probe_out(det, H)[:, 0] for k, (H, L, y) in FE.items()}        # back-end score (positive = fake)
print(f"back-end: best epoch {det_ep}, dev mean EER {min(det_hist):.2f}% (original model {dev_mean_eer(O):.2f}%)")
print("layer weights learned by the back-end:", np.round(torch.softmax(det.a, 0).detach().cpu().numpy(), 3).tolist())

o_dev = np.concatenate([O[k] for k in DEV_NAMES]); p_dev = np.concatenate([P[k] for k in DEV_NAMES])
NORM = dict(mo=float(o_dev.mean()), so=float(o_dev.std()), mp=float(p_dev.mean()), sp=float(p_dev.std()))
def fuse(o, p, w):
    return (1 - w) * (o - NORM["mo"]) / NORM["so"] + w * (p - NORM["mp"]) / NORM["sp"]
GRID = [round(float(w), 2) for w in np.linspace(0, 1, 21)]
fusion = pd.DataFrame([{"w": w, "dev_mean_EER": dev_mean_eer({k: fuse(O[k], P[k], w) for k in DEV_NAMES})}
                       for w in GRID])
W = float(fusion.sort_values(["dev_mean_EER", "w"]).iloc[0]["w"])   # ties -> the smaller w (closer to the original)
fusion.to_csv(f"{WORK}/step8c_fusion.csv", index=False)
print(fusion.iloc[::2].round(2).to_string(index=False))
print(f"chosen fusion weight w = {W}  (0 = original model only, 1 = back-end only)")
S = {"original": O, "back-end": P, "fused": {k: fuse(O[k], P[k], W) for k in FE}}
dev_tab = pd.DataFrame({tag: {k: eer(FE[k][2], sc[k]) for k in DEV_NAMES} for tag, sc in S.items()})
dev_tab.loc["MEAN"] = dev_tab.mean()
print("\nDev EER % (the data all choices were made on):")
print(dev_tab.round(2).to_string())


In [ ]:
# CELL 9: processing-aware calibration ("automatic line profile"). A second small probe on the same frozen
# features recognises which processing the audio went through; every processing has its own threshold,
# calibrated on held-out speakers. This corrects score SHIFTS (e.g. Opus); it cannot undo information loss.
ctr = torch.from_numpy(Ctr).to(DEVICE)
CAL_SETS = {"clean": ["held-out speakers, clean", "Commercial, unseen Hume", "In-the-Wild (dev sample)"],
            **{c: [k for k in list(DEV) + list(EXTRA) if k.startswith("held-out") and cond_of(k) == c]
               for c in CONDS[1:]}}
assert all(CAL_SETS[c] for c in CONDS), CAL_SETS
CAL_NAMES = sorted({k for v in CAL_SETS.values() for k in v})
cond_true = {k: CONDS.index(cond_of(k)) for k in FE}

def cond_metric(probe):
    return -float(np.mean([(probe_out(probe, FE[k][0]).argmax(1) == cond_true[k]).mean() for k in CAL_NAMES]))

cls, cls_ep, cls_hist = train_probe(len(CONDS), ctr, cond_metric, epochs=20)
PRED = {k: probe_out(cls, FE[k][0]).argmax(1) for k in FE}
acc = {k: 100 * float((PRED[k] == cond_true[k]).mean()) for k in FE}
print(f"processing classifier: best epoch {cls_ep}, mean accuracy on held-out speakers {-100 * min(cls_hist):.1f}%")
for k in TEST:
    top = int(np.bincount(PRED[k], minlength=len(CONDS)).argmax())
    print(f"  {k:46s} true: {cond_of(k):30s} accuracy {acc[k]:5.1f}%  (most often predicted: {CONDS[top]})")

def thr_at_eer(names, sc):
    y = np.concatenate([FE[k][2] for k in names]); s = np.concatenate([sc[k] for k in names])
    return float(eer_auc(y, s)["threshold_at_EER"])

THR = {tag: {"single": thr_at_eer(DEV_NAMES, sc), **{c: thr_at_eer(CAL_SETS[c], sc) for c in CONDS}}
       for tag, sc in S.items()}
print("\nOriginal model thresholds (llr): single", round(THR["original"]["single"], 2), "| per processing:",
      {c: round(THR["original"][c], 2) for c in CONDS})

def rates(y, s, thr):
    """% of REAL clips flagged (false alarms) and % of FAKE clips missed, at a threshold (scalar or per clip)."""
    y = np.asarray(y); flag = np.asarray(s) >= thr
    return 100 * float(flag[y == 0].mean()), 100 * float((~flag[y == 1]).mean())


In [ ]:
# CELL 10: final test (every set unseen): original vs back-end vs fused; one threshold vs automatic per-processing
rows = []
for k in TEST:
    y = FE[k][2]
    for tag, sc in S.items():
        s, r = sc[k], eer_auc(y, sc[k])
        fa1, mr1 = rates(y, s, THR[tag]["single"])
        fa2, mr2 = rates(y, s, np.array([THR[tag][CONDS[c]] for c in PRED[k]]))
        fa3, mr3 = rates(y, s, THR[tag][cond_of(k)])
        rows.append({"test set": k, "scorer": tag, "EER_%": r["EER_%"], "AUC": r["AUC"],
                     "FAR_single_%": fa1, "MR_single_%": mr1, "FAR_auto_%": fa2, "MR_auto_%": mr2,
                     "FAR_oracle_%": fa3, "MR_oracle_%": mr3, "processing_accuracy_%": acc[k]})
res = pd.DataFrame(rows).round(3)
res.to_csv(f"{WORK}/step8c_results.csv", index=False)
shift = pd.DataFrame([{"test set": k, "median_llr_real": float(np.median(O[k][FE[k][2] == 0])),
                       "median_llr_fake": float(np.median(O[k][FE[k][2] == 1]))} for k in TEST]).round(2)
shift.to_csv(f"{WORK}/step8c_shift.csv", index=False)

pd.set_option("display.width", 250)
print("EER % on unseen test sets (lower is better):")
print(res.pivot(index="test set", columns="scorer", values="EER_%").reindex(list(TEST))[list(S)].round(2).to_string())
print("\nOriginal model, false alarms on REAL voices / missed FAKES (%), one threshold -> automatic per-processing:")
o = res[res.scorer == "original"].set_index("test set")
print(o[["FAR_single_%", "FAR_auto_%", "MR_single_%", "MR_auto_%", "processing_accuracy_%"]].round(1).to_string())
print("\nFull table:"); print(res.to_string(index=False))

dsp_rows = [k for k in TEST if cond_of(k) != "clean"]
def m(tag, col, names):
    return float(res[(res.scorer == tag) & res["test set"].isin(names)][col].mean())
summary = {
    "fusion_w": W, "backend_best_epoch": det_ep, "processing_classifier_best_epoch": cls_ep,
    "dev_mean_EER": {tag: float(dev_tab.loc["MEAN", tag]) for tag in S},
    "test_mean_EER_all": {tag: m(tag, "EER_%", list(TEST)) for tag in S},
    "test_mean_EER_processed": {tag: m(tag, "EER_%", dsp_rows) for tag in S},
    "processed_test_FAR_single_vs_auto": {tag: [m(tag, "FAR_single_%", dsp_rows), m(tag, "FAR_auto_%", dsp_rows)] for tag in S},
    "processed_test_MR_single_vs_auto": {tag: [m(tag, "MR_single_%", dsp_rows), m(tag, "MR_auto_%", dsp_rows)] for tag in S},
    "processing_accuracy_heldout_%": -100 * min(cls_hist),
    "processing_accuracy_test_%": {k: acc[k] for k in TEST},
    "thresholds_llr_original": THR["original"], "thresholds_fused": THR["fused"],
    "backend_layer_weights": torch.softmax(det.a, 0).detach().cpu().numpy().round(4).tolist(),
    "classifier_layer_weights": torch.softmax(cls.a, 0).detach().cpu().numpy().round(4).tolist(),
    "conds": CONDS, "views_per_clip": VIEWS, "train_views": int(N),
}
json.dump(summary, open(f"{WORK}/step8c_summary.json", "w"), indent=2)
np.savez(f"{WORK}/step8c_backend.npz", conds=np.array(CONDS), mu=MU.cpu().numpy(), sd=SD.cpu().numpy(),
         det_a=det.a.detach().cpu().numpy(), det_w=det.lin.weight.detach().cpu().numpy()[0],
         det_b=det.lin.bias.detach().cpu().numpy(), cls_a=cls.a.detach().cpu().numpy(),
         cls_w=cls.lin.weight.detach().cpu().numpy(), cls_b=cls.lin.bias.detach().cpu().numpy(),
         fusion_w=W, mo=NORM["mo"], so=NORM["so"], mp=NORM["mp"], sp=NORM["sp"],
         thr_original=np.array([THR["original"][c] for c in CONDS]), thr_fused=np.array([THR["fused"][c] for c in CONDS]),
         thr_single_original=THR["original"]["single"], thr_single_fused=THR["fused"]["single"])
print("\n==== SUMMARY (please copy everything below) ====")
for tag in S:
    print(f"{tag:9s} dev mean EER {summary['dev_mean_EER'][tag]:6.2f}% | test mean EER all {summary['test_mean_EER_all'][tag]:6.2f}%"
          f" | processed {summary['test_mean_EER_processed'][tag]:6.2f}% | false alarms on processed sets "
          f"{summary['processed_test_FAR_single_vs_auto'][tag][0]:.1f}% -> {summary['processed_test_FAR_single_vs_auto'][tag][1]:.1f}%"
          f" | misses {summary['processed_test_MR_single_vs_auto'][tag][0]:.1f}% -> {summary['processed_test_MR_single_vs_auto'][tag][1]:.1f}%")
print(f"fusion weight w = {W}; processing classifier accuracy on held-out speakers {summary['processing_accuracy_heldout_%']:.1f}%")


In [ ]:
# CELL 11: export the SAME detector with its layer averages as a second output, for the live server
# (the back-end, the fusion and the processing classifier then run in NumPy from step8c_backend.npz).
EXPORT_ONNX = True
if EXPORT_ONNX:
    import torch.nn.utils.parametrize as _P
    m_cpu = model.float().cpu().eval()
    for mod in [mod for mod in m_cpu.modules() if _P.is_parametrized(mod)]:
        for pname in list(mod.parametrizations.keys()):
            _P.remove_parametrizations(mod, pname, leave_parametrized=True)
    for mod in [mod for mod in m_cpu.modules() if hasattr(mod, "weight_g")]:
        torch.nn.utils.remove_weight_norm(mod)
    class LayerExport(nn.Module):
        """Same maths as the detector; layer_norm over a dynamic length written as mean/var for ONNX."""
        def __init__(self, m):
            super().__init__()
            self.m = m
        def forward(self, wav):
            wav = (wav - wav.mean(-1, keepdim=True)) / torch.sqrt(wav.var(-1, keepdim=True, unbiased=False) + 1e-5)
            out = self.m.ssl(wav, output_hidden_states=True)
            layers = torch.stack([h.mean(1) for h in out.hidden_states[:-1]] + [out.last_hidden_state.mean(1)], 1)
            return self.m.head(layers[:, -1]), layers
    onnx_path = f"{WORK}/voiceshield_layers.onnx"
    kw = dict(input_names=["wav"], output_names=["logits", "layers"], opset_version=17,
              dynamic_axes={"wav": {0: "batch"}, "logits": {0: "batch"}, "layers": {0: "batch"}})
    dummy = torch.randn(1, T) * 0.1
    try:
        torch.onnx.export(LayerExport(m_cpu).eval(), dummy, onnx_path, dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(LayerExport(m_cpu).eval(), dummy, onnx_path, **kw)
    import onnxruntime as ort
    sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    x = np.stack([crop(w) for w in TEST["test speakers, clean"][0][:4]]).astype(np.float32)
    xn = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-7)   # what the live server sends
    with torch.no_grad():
        ref_lo, ref_la = LayerExport(m_cpu).eval()(torch.from_numpy(x))
    lo, la = sess.run(None, {"wav": xn})
    print(f"Exported {onnx_path} ({os.path.getsize(onnx_path) / 1e9:.2f} GB); max difference vs PyTorch: "
          f"logits {np.abs(lo - ref_lo.numpy()).max():.4f}, layers {np.abs(la - ref_la.numpy()).max():.4f}")
